In [1]:
import os
import shutil
import glob
import skimage.io

import torch
import numpy as np
from pytorch3d.structures import Volumes
from pytorch3d.transforms import so3_exp_map
from pytorch3d.renderer import (
    FoVPerspectiveCameras,
    NDCMultinomialRaysampler, 
    VolumeRenderer,
    look_at_view_transform,
)

from monai.utils import first, set_determinism, get_seed, MAX_SEED

import matplotlib.pyplot as plt
%matplotlib inline 
from data import *
from raymarcher import *

In [ ]:
class Hparams:
    def __init__(self, datadir, shape, batch_size, seed) -> None:
        self.datadir = datadir
        self.shape = shape
        self.batch_size = batch_size
        self.seed = seed

# hparams = parser.parse_args()
hparams = Hparams(
    datadir='data',
    shape=256, 
    batch_size=4, 
    seed=42
)
# Create data module
train_image3d_folders = [
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/Verse2019/raw/train/rawdata/'),
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/Verse2020/raw/train/rawdata/'),
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/Verse2019/raw/val/rawdata/'),
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/Verse2020/raw/val/rawdata/'),
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/Verse2019/raw/test/rawdata/'),
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/Verse2020/raw/test/rawdata/'),

    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/UWSpine/processed/train/images'),
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/UWSpine/processed/test/images/'),

    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/NSCLC/processed/train/images'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/MOSMED/processed/train/images/CT-0'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/MOSMED/processed/train/images/CT-1'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/MOSMED/processed/train/images/CT-2'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/MOSMED/processed/train/images/CT-3'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/MOSMED/processed/train/images/CT-4'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/Imagenglab/processed/train/images'),
]
train_label3d_folders = [

]

train_image2d_folders = [
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/JSRT/processed/images/'), 
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/ChinaSet/processed/images/'), 
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/Montgomery/processed/images/'),
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/VinDr/v1/processed/train/images/'), 
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/VinDr/v1/processed/test/images/'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/T62020/20200501/raw/images'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/T62021/20211101/raw/images'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/VinDr/v1/processed/train/images/'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/VinDr/v1/processed/test/images/'), 
]
train_label2d_folders = [
]

val_image3d_folders = train_image3d_folders
val_image2d_folders = [
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/JSRT/processed/images/'), 
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/ChinaSet/processed/images/'), 
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/Montgomery/processed/images/'),
    # os.path.join(hparams.datadir, 'ChestXRLungSegmentation/VinDr/v1/processed/train/images/'), 
    os.path.join(hparams.datadir, 'ChestXRLungSegmentation/VinDr/v1/processed/test/images/'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/T62020/20200501/raw/images'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/T62021/20211101/raw/images'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/VinDr/v1/processed/train/images/'), 
    # os.path.join(hparams.datadir, 'SpineXRVertSegmentation/VinDr/v1/processed/test/images/'), 
]

test_image3d_folders = val_image3d_folders
test_image2d_folders = val_image2d_folders

datamodule = NeRVDataModule(
    train_image3d_folders = train_image3d_folders, 
    train_image2d_folders = train_image2d_folders, 
    val_image3d_folders = val_image3d_folders, 
    val_image2d_folders = val_image2d_folders, 
    test_image3d_folders = test_image3d_folders, 
    test_image2d_folders = test_image2d_folders, 
    batch_size = hparams.batch_size, 
    shape = hparams.shape
)
datamodule.setup(seed=hparams.seed)


In [ ]:
class VolumeModel(torch.nn.Module):
    def __init__(self, renderer, volume_size=[64] * 3, voxel_size=0.1):
        super().__init__()
        # After evaluating torch.sigmoid(self.log_colors), we get 
        # densities close to zero.
        # self.log_densities = torch.nn.Parameter(-4.0 * torch.ones(1, *volume_size))
        # # After evaluating torch.sigmoid(self.log_colors), we get 
        # # a neutral gray color everywhere.
        # self.log_colors = torch.nn.Parameter(torch.zeros(3, *volume_size))
        self._voxel_size = voxel_size
        # Store the renderer module as well.
        self._renderer = renderer
        
    def forward(self, image3d, cameras):
        batch_size = cameras.R.shape[0]

        # Convert the log-space values to the densities/colors
        # densities = torch.sigmoid(self.log_densities)
        # colors = torch.sigmoid(self.log_colors)
        
        # Instantiate the Volumes object, making sure
        # the densities and colors are correctly
        # expanded batch_size-times.
        features = image3d.view(-1, 1, 1, 1, 1).repeat(batch_size, 3, 1, 1, 1)
        # features = radiances[:,[0]].repeat(1, 3, 1, 1, 1)
        # densities = radiances[:,[1]]
        densities = torch.zeros_like(image3d.view(-1, 1, 1, 1, 1)).repeat(batch_size, 1, 1, 1, 1)

        volumes = Volumes(
            densities = densities* .02 + .03,
            features = features,
            voxel_size=self._voxel_size,
        )
        
        # Given cameras and volumes, run the renderer
        # and return only the first output value 
        # (the 2nd output is a representation of the sampled
        # rays which can be omitted for our purpose).
        return self._renderer(cameras=cameras, volumes=volumes)[0]
    
# # A helper function for evaluating the smooth L1 (huber) loss
# # between the rendered silhouettes and colors.
# def huber(x, y, scaling=0.1):
#     diff_sq = (x - y) ** 2
#     loss = ((1 + diff_sq / (scaling**2)).clamp(1e-4).sqrt() - 1) * float(scaling)
#     return loss
    
def generate_xray_renders(
    device: torch.device = torch.device("cpu"), 
    image3d: torch.Tensor = torch.zeros([256, 256, 256]), 
    num_views: int = 40, 
    azimuth_range: float = 180,
):
    """
    This function generates `num_views` renders of a cow mesh.
    The renders are generated from viewpoints sampled at uniformly distributed
    azimuth intervals. The elevation is kept constant so that the camera's
    vertical position coincides with the equator.

    For a more detailed explanation of this code, please refer to the
    docs/tutorials/fit_textured_mesh.ipynb notebook.

    Args:
        azimuth_range: number of degrees on each side of the start position to
            take samples

    Returns:
        cameras: A batch of `num_views` `FoVPerspectiveCameras` from which the
            images are rendered.
        images: A tensor of shape `(num_views, height, width, 3)` containing
            the rendered images.
    """

    # set the paths

    # # Setup
    # if torch.cuda.is_available():
    #     device = torch.device("cuda:0")
    #     torch.cuda.set_device(device)
    # else:
    #     device = torch.device("cpu")

    
    # Get a batch of viewing angles.
    elev = torch.linspace(0, 0, num_views).to(device)  # keep constant
    azim = torch.linspace(-azimuth_range, azimuth_range, num_views).to(device) + 180.0

    
    # Initialize an OpenGL perspective camera that represents a batch of different
    # viewing angles. All the cameras helper methods support mixed type inputs and
    # broadcasting. So we can view the camera from the a distance of dist=2.7, and
    # then specify elevation and azimuth angles for each viewpoint as tensors.
    R, T = look_at_view_transform(dist=2.7, elev=elev, azim=azim)
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T).to(device)

    # Our rendered scene is centered around (0,0,0) 
    # and is enclosed inside a bounding box
    # whose side is roughly equal to 3.0 (world units).
    volume_extent_world = 3.0

    # 1) Instantiate the raysampler.
    # Here, NDCMultinomialRaysampler generates a rectangular image
    # grid of rays whose coordinates follow the PyTorch3D
    # coordinate conventions.
    # Since we use a volume of size 128^3, we sample n_pts_per_ray=150,
    # which roughly corresponds to a one ray-point per voxel.
    # We further set the min_depth=0.1 since there is no surface within
    # 0.1 units of any camera plane.
    raysampler = NDCMultinomialRaysampler(
        image_width=256,
        image_height=256,
        n_pts_per_ray=512,
        min_depth=0.1,
        max_depth=volume_extent_world,
    )

    # 2) Instantiate the raymarcher.
    # Here, we use the standard EmissionAbsorptionRaymarcher 
    # which marches along each ray in order to render
    # each ray into a single 3D color vector 
    # and an opacity scalar.
    raymarcher = XRayEmissionAbsorptionRaymarcher()

    # Finally, instantiate the volumetric render
    # with the raysampler and raymarcher objects.
    renderer = VolumeRenderer(
        raysampler=raysampler, 
        raymarcher=raymarcher,
    )

    # Instantiate the volumetric model.
    # We use a cubical volume with the size of 
    # one side = 128. The size of each voxel of the volume 
    # is set to volume_extent_world / volume_size s.t. the
    # volume represents the space enclosed in a 3D bounding box
    # centered at (0, 0, 0) with the size of each side equal to 3.
    volume_size = 256
    voxel_size = volume_extent_world / volume_size
    # volume_model = VolumeModel(
    #     renderer,
    #     volume_size=[volume_size] * 3, 
    #     voxel_size = volume_extent_world / volume_size,
    # ).to(device)
    batch_size = cameras.R.shape[0]

    features = image3d.view(-1, 1, 256, 256, 256).repeat(batch_size, 3, 1, 1, 1)
    # features = radiances[:,[0]].repeat(1, 3, 1, 1, 1)
    # densities = radiances[:,[1]]
    densities = torch.zeros_like(image3d.view(-1, 1, 256, 256, 256)).repeat(batch_size, 1, 1, 1, 1)

    volumes = Volumes(
        densities = densities * .02,
        features = features,
        voxel_size=voxel_size,
    )
    screen_RGBA, ray_bundles = renderer(cameras=cameras, volumes=volumes) #[...,:3]
    # # rays_points = ray_bundle_to_ray_points(ray_bundles)

    screen_RGBA = screen_RGBA.permute(0, 3, 2, 1) # B H W C to B C H W
    screen_RGB = screen_RGBA[:, :3].mean(dim=1, keepdim=True)
    
    # Render the cow mesh from each viewing angle
    # target_images = volume_model(image3d, cameras=cameras)
    return cameras, screen_RGB

In [ ]:
# target_cameras, target_images, target_silhouettes = generate_cow_renders(num_views=40, azimuth_range=180)
# print(f'Generated {len(target_images)} images/silhouettes/cameras.')
DEVICE = torch.device("cpu")
BATCH_SIZE = 40
debug_data = first(datamodule.train_dataloader())
image3d, image2d = (debug_data["image3d"][0][0], debug_data["image2d"][0][0])
print(f"image3d shape: {image3d.shape}, \
        image2d shape: {image2d.shape}")
image3d = image3d.to(DEVICE)

target_cameras, target_images = generate_xray_renders(
        DEVICE, 
        image3d, 
        num_views=BATCH_SIZE, 
        azimuth_range=180
)
print(f'Generated {len(target_images)} images/silhouettes/cameras.')

In [ ]:
print(target_images.shape)
shutil.rmtree("samples", ignore_errors=True)
os.makedirs("samples")
for k in range(BATCH_SIZE):
    fig = plt.figure(figsize=(5, 5))
    image = target_images[k].squeeze().detach().cpu().numpy()
    image /= image.max() 
    image *= 255
    # plt.imshow(image.astype(np.uint8), cmap=plt.cm.gray)
    # plt.axis("off")
    # plt.show()
    skimage.io.imsave(os.path.join("samples", str(k).zfill(3)+".png"), image)
    